# 02 — Dataset audit (Faz 2)

**`01_download` ile aynı oturumda çalıştır** — ham veri `/content/raw` altında. Oturum koptuysa önce 01'in 1. ve 2. bölümlerini tekrar çalıştır.

Bu notebook FASDD_UAV'yi ölçer ve D-Fire negatiflerini kopya kontrolüne katar. Boreal, Faz 3'te YOLO formatına çevrildikten sonra aynı ölçümlerden geçecek.

Çıktılar (`firesmoke/reports/`): `audit_v1.md`, `audit_hist.png`, `audit_samples_*.jpg`, `audit_duplicates.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install imagehash tabulate

In [ ]:
import json, random
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/firesmoke')
RAW  = Path('/content/raw')
REP  = ROOT / 'reports'
REP.mkdir(exist_ok=True)
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp'}
NAMES = {0: 'fire', 1: 'smoke'}
random.seed(0)

## 1. Dosyaları eşleştir

In [ ]:
def find_dir(base, name):
    hits = [p for p in base.rglob(name) if p.is_dir()]
    assert hits, f'{name} bulunamadı — 01_download çalıştırıldı mı?'
    return min(hits, key=lambda p: len(p.parts))

F = RAW / 'fasdd_uav'
IMG_DIR  = find_dir(F, 'images')
YOLO_DIR = find_dir(F, 'YOLO_UAV')
LBL_DIR  = YOLO_DIR / 'labels'
img_by_stem = {p.stem: p for p in IMG_DIR.rglob('*')
               if p.suffix.lower() in IMG_EXT and not p.name.startswith('._')}

split_of = {}
for s in ['train', 'val', 'test']:
    for line in (YOLO_DIR / f'{s}.txt').read_text().split():
        split_of[Path(line).stem] = s
print(len(img_by_stem), 'görüntü,', len(split_of), 'split kaydı')
print('split kaydı olmayan görüntü:', len(set(img_by_stem) - set(split_of)))

In [ ]:
def img_size(p):
    try:
        with Image.open(p) as im:
            return im.size
    except Exception:
        return None

with ThreadPoolExecutor(16) as ex:
    sizes = dict(zip(img_by_stem, ex.map(img_size, img_by_stem.values())))
print('açılamayan görüntü:', sum(v is None for v in sizes.values()))

## 2. Etiketleri tabloya dök

Her kutu bir satır. YOLO formatı normalize (0–1); piksel boyutunu görüntü çözünürlüğüyle çarparak buluyoruz. `side_px` = √(genişlik × yükseklik): kutunun "kenar uzunluğu" gibi düşün.

In [ ]:
img_rows, box_rows, bad_lines = [], [], []
for stem, p in img_by_stem.items():
    if sizes[stem] is None:
        continue
    W, H = sizes[stem]
    lp = LBL_DIR / f'{stem}.txt'
    lines = [l for l in lp.read_text().splitlines() if l.strip()] if lp.exists() else []
    img_rows.append({'stem': stem, 'split': split_of.get(stem, '?'), 'W': W, 'H': H,
                     'n_boxes': len(lines), 'has_label_file': lp.exists()})
    for l in lines:
        parts = l.split()
        if len(parts) != 5:
            bad_lines.append((stem, l))
            continue
        c = int(float(parts[0]))
        x, y, w, h = map(float, parts[1:])
        box_rows.append({'stem': stem, 'split': split_of.get(stem, '?'), 'cls': c,
                         'x': x, 'y': y, 'w': w, 'h': h, 'W': W, 'H': H})

imgs = pd.DataFrame(img_rows)
boxes = pd.DataFrame(box_rows)
boxes['name'] = boxes.cls.map(NAMES).fillna('BILINMEYEN')
boxes['side_px'] = np.sqrt(boxes.w * boxes.W * boxes.h * boxes.H)

coords = boxes[['x', 'y', 'w', 'h']]
print('kutu sayısı           :', len(boxes))
print('hatalı satır          :', len(bad_lines), bad_lines[:3])
print('koordinat 0–1 dışında :', int(((coords < 0) | (coords > 1)).any(axis=1).sum()))
print('sıfır alanlı kutu     :', int(((boxes.w <= 0) | (boxes.h <= 0)).sum()))
print('bilinmeyen sınıf      :', sorted(set(boxes.cls) - set(NAMES)))

## 3. Sınıf ve split dağılımı

In [ ]:
SPLITS = ['train', 'val', 'test']
t_boxes = (boxes.pivot_table(index='split', columns='name', values='stem', aggfunc='count', fill_value=0)
                .reindex(SPLITS))

cls_per_img = boxes.groupby('stem').cls.agg(frozenset)
CAT = {frozenset(): 'negatif', frozenset({0}): 'sadece fire', frozenset({1}): 'sadece smoke'}
imgs['cat'] = [CAT.get(cls_per_img.get(s, frozenset()), 'fire+smoke') for s in imgs.stem]
t_imgs = (imgs.pivot_table(index='split', columns='cat', values='stem', aggfunc='count', fill_value=0)
              .reindex(SPLITS))
t_imgs['negatif %'] = (100 * t_imgs.get('negatif', 0) / t_imgs.sum(axis=1)).round(1)

print('Kutular:'); display(t_boxes)
print('Görüntüler:'); display(t_imgs)

## 4. Kutu boyutu — `imgsz` kararını veren tablo

YOLO görüntüyü uzun kenarı `imgsz` olacak şekilde küçültür; kutular da aynı oranda küçülür. Aşağıdaki tablolar **küçültme sonrası** kutu kenarının dağılımını gösterir. 16 px altına düşen kutular modelin pratikte zor gördüğü bölgedir.

In [ ]:
BINS, LABELS = [0, 8, 16, 32, 96, np.inf], ['<8', '8-16', '16-32', '32-96', '>96']

def size_table(df, imgsz):
    side = df.side_px * imgsz / df[['W', 'H']].max(axis=1)
    b = pd.cut(side, BINS, labels=LABELS, right=False).rename('kenar_px')
    t = pd.crosstab(df.name, b, normalize='index', dropna=False).mul(100).round(1)
    return t.reindex(columns=LABELS, fill_value=0)

print('Orijinal çözünürlükte kutu kenarı (px):')
display(boxes.groupby('name').side_px.describe(percentiles=[.1, .25, .5, .75, .9]).round(1))
size_tables = {sz: size_table(boxes, sz) for sz in [640, 1024, 1280]}
for sz, t in size_tables.items():
    print(f'\nimgsz={sz} sonrası kutu kenarı (% satır):')
    display(t)

In [ ]:
res = imgs.groupby(['W', 'H']).size().sort_values(ascending=False)
print('farklı çözünürlük sayısı:', len(res))
display(res.head(10).to_frame('görüntü'))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, g in boxes.groupby('name'):
    ax[0].hist(np.log2(g.side_px.clip(lower=1)), bins=40, alpha=.6, label=name)
ax[0].set_xlabel('log2(kutu kenarı, px)')
ax[0].set_title('Kutu boyutu (orijinal çözünürlük)')
ax[0].legend()
ax[1].hist(imgs.n_boxes, bins=range(0, int(imgs.n_boxes.max()) + 2))
ax[1].set_yscale('log')
ax[1].set_title('Görüntü başına kutu sayısı')
plt.tight_layout(); plt.savefig(REP / 'audit_hist.png', dpi=110); plt.show()

## 5. Bozuk dosya kontrolü

In [ ]:
def verify(p):
    try:
        with Image.open(p) as im:
            im.verify()
        return None
    except Exception as e:
        return (p.name, str(e)[:80])

with ThreadPoolExecutor(16) as ex:
    corrupt = [r for r in ex.map(verify, img_by_stem.values()) if r]
print('bozuk:', len(corrupt), corrupt[:5])

## 6. Yakın kopya kontrolü (sızıntı)

Her görüntünün perceptual hash'i (phash, 64 bit) hesaplanır; iki hash arasında en fazla `THR` bit fark varsa görüntüler neredeyse aynıdır. FASDD_UAV kısmen video karelerinden geldiği için **resmi split'ler arasında kopya** olabilir: bu, test skorunu şişiren sızıntıdır. FASDD ↔ D-Fire kopyaları da raporlanır.

Süre: ~35 bin görüntü için birkaç dakika.

In [ ]:
import imagehash

NEG = RAW / 'dfire_neg' / 'images'
pool = [(s, 'fasdd_uav', p) for s, p in img_by_stem.items()]
if NEG.exists():
    pool += [(p.stem, 'dfire_neg', p) for p in sorted(NEG.iterdir()) if not p.name.startswith('._')]

def ph(p):
    try:
        with Image.open(p) as im:
            return int(str(imagehash.phash(im)), 16)
    except Exception:
        return None

with ThreadPoolExecutor(16) as ex:
    hashes = list(ex.map(ph, [p for _, _, p in pool]))

keep = [i for i, h in enumerate(hashes) if h is not None]
meta = pd.DataFrame([{'stem': pool[i][0], 'source': pool[i][1]} for i in keep])
meta['split'] = [split_of.get(s, '?') if src == 'fasdd_uav' else 'neg'
                 for s, src in zip(meta.stem, meta.source)]
Hs = np.array([hashes[i] for i in keep], dtype=np.uint64)
print(len(Hs), 'hash hesaplandı')

In [ ]:
_POP = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)

def popcount64(x):
    if hasattr(np, 'bitwise_count'):
        return np.bitwise_count(x)
    x = np.ascontiguousarray(x)
    return _POP[x.view(np.uint8).reshape(*x.shape, 8)].sum(-1)

THR, CH = 4, 512
found = []
for i in range(0, len(Hs), CH):
    d = popcount64(Hs[i:i + CH, None] ^ Hs[None, :])
    la, b = np.nonzero(d <= THR)
    m = (la + i) < b                      # her çifti bir kez say, kendisiyle eşleşmeyi at
    found.append(np.stack([la[m] + i, b[m], d[la[m], b[m]]], axis=1))
pairs = np.concatenate(found) if found else np.zeros((0, 3), dtype=int)

dup = pd.DataFrame(pairs, columns=['i', 'j', 'dist'])
dup = dup.join(meta.add_suffix('_i'), on='i').join(meta.add_suffix('_j'), on='j')
both_fasdd = (dup.source_i == 'fasdd_uav') & (dup.source_j == 'fasdd_uav')
cross_split = dup[both_fasdd & (dup.split_i != dup.split_j)]
cross_source = dup[dup.source_i != dup.source_j]

print('toplam yakın-kopya çifti      :', len(dup))
print('FASDD içinde, farklı split    :', len(cross_split))
display(cross_split.groupby(['split_i', 'split_j']).size().to_frame('çift'))
print('FASDD ↔ D-Fire negatif        :', len(cross_source))
test_leak = set(cross_split[cross_split.split_i == 'test'].stem_i) | set(cross_split[cross_split.split_j == 'test'].stem_j)
print('kopyası train/val\'de olan test görüntüsü:', len(test_leak))
dup.to_csv(REP / 'audit_duplicates.csv', index=False)

## 7. Görsel kontrol

Kutuları görüntü üzerinde gör. Kırmızı = fire, camgöbeği = smoke. Her grid için şu soruları cevapla ve **cevaplarını bana yaz**:

1. Kutular doğru yerde mi?
2. Duman nasıl kutulanmış: bütün bulutu saran tek büyük kutu mu, parça parça mı?
3. Kamera açısı: tam tepeden (nadir) mi, eğik mi? Görüntülerin üstünde gökyüzü var mı?
4. Mum, kibrit, şömine gibi drone senaryosuna uymayan alevler var mı?
5. D-Fire negatifleri: dış mekan mı (bulut, sis, gün batımı) yoksa iç mekan mı?

In [ ]:
box_groups = {s: g for s, g in boxes.groupby('stem')}
COLORS = {0: 'red', 1: 'cyan'}

def draw_grid(paths, title, fname, n=30, cols=6, with_boxes=True):
    paths = random.sample(paths, min(n, len(paths)))
    if not paths:
        print('örnek yok:', title); return
    rows = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 2.6), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, p in zip(axes.ravel(), paths):
        im = Image.open(p).convert('RGB'); W, H = im.size
        ax.imshow(im)
        if with_boxes and p.stem in box_groups:
            for b in box_groups[p.stem].itertuples():
                ax.add_patch(plt.Rectangle(((b.x - b.w / 2) * W, (b.y - b.h / 2) * H), b.w * W, b.h * H,
                                           fill=False, color=COLORS.get(b.cls, 'yellow'), lw=1.5))
        ax.set_title(p.stem[:24], fontsize=7)
    fig.suptitle(title)
    plt.tight_layout(); plt.savefig(REP / fname, dpi=90); plt.show()

pos = [img_by_stem[s] for s in imgs[imgs.cat != 'negatif'].stem]
draw_grid(pos, 'FASDD_UAV — rastgele pozitifler', 'audit_samples_pos.jpg')

In [ ]:
tiny = boxes[boxes.side_px * 640 / boxes[['W', 'H']].max(axis=1) < 16].stem.unique().tolist()
draw_grid([img_by_stem[s] for s in tiny], "imgsz=640'ta 16 px altına düşen kutular", 'audit_samples_tiny.jpg', n=18)

In [ ]:
if NEG.exists():
    draw_grid(sorted(NEG.iterdir()), 'D-Fire — rastgele negatifler', 'audit_samples_dfire_neg.jpg', n=24, with_boxes=False)

## 8. Rapor → Drive

In [ ]:
lines = [
    '# Audit v1 — FASDD_UAV', '',
    f'- Görüntü: {len(imgs)} · kutu: {len(boxes)} · bozuk: {len(corrupt)} · hatalı satır: {len(bad_lines)}',
    f'- Farklı çözünürlük: {len(res)} · en yaygın: {res.index[0][0]}x{res.index[0][1]} ({res.iloc[0]} görüntü)',
    f'- Yakın kopya çifti (phash ≤ {THR} bit): {len(dup)} · FASDD split arası: {len(cross_split)}'
    f' · kopyası train/val\'de olan test görüntüsü: {len(test_leak)} · FASDD↔D-Fire: {len(cross_source)}',
    '', '## Kutular (split × sınıf)', t_boxes.to_markdown(),
    '', '## Görüntüler (split × kategori)', t_imgs.to_markdown(),
    '', '## Orijinal kutu kenarı (px)', boxes.groupby('name').side_px.describe(percentiles=[.1, .5, .9]).round(1).to_markdown(),
]
for sz, t in size_tables.items():
    lines += ['', f'## imgsz={sz} sonrası kutu kenarı (%)', t.to_markdown()]
(REP / 'audit_v1.md').write_text('\n'.join(lines))
print('\n'.join(lines))

**Kontrol noktası — bana gönderilecekler:**
1. Yukarıdaki raporun tamamı (`audit_v1.md`)
2. Görsel kontroldeki 5 sorunun cevabı
3. 01'deki Boreal format çıktısı

Bunlarla `03_merge_split.ipynb`'yi (Boreal dönüştürücü, sızıntısız split, negatif oranı, `data.yaml`) senin verine göre yazacağım.